<a href="https://colab.research.google.com/github/AneleGMthembu/Book-Haven/blob/main/BERT-CNN%20HYBRID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving final_feature_table.csv to final_feature_table.csv


In [3]:
!pip install transformers torch scikit-learn pandas numpy tensorflow

In [4]:
import pandas as pd

data = pd.read_csv("final_feature_table.csv")
print("Total rows:", len(data))
print("Rows with ad_network_score:", data["ad_network_score"].notna().sum())
print("Label balance:\n", data["label"].value_counts())

model_data = data.dropna(subset=["ad_network_score", "clickbait_score", "share_count", "title"]).reset_index(drop=True)
print("\nRows usable for the full three-feature model:", len(model_data))
print(model_data["label"].value_counts())

Total rows: 22991
Rows with ad_network_score: 3084
Label balance:
 label
0    17305
1     5686
Name: count, dtype: int64

Rows usable for the full three-feature model: 3084
label
0    2412
1     672
Name: count, dtype: int64


In [5]:
import torch
from transformers import BertTokenizer, BertModel
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()  # inference mode, not training BERT itself

def get_bert_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = bert_model(**inputs)
        # Use the [CLS] token embedding as the sentence representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
        print(f"Processed {min(i+batch_size, len(texts))}/{len(texts)}")
    return np.vstack(all_embeddings)

bert_embeddings = get_bert_embeddings(model_data["title"])
print("\nBERT embeddings shape:", bert_embeddings.shape)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processed 32/3084
Processed 64/3084
Processed 96/3084
Processed 128/3084
Processed 160/3084
Processed 192/3084
Processed 224/3084
Processed 256/3084
Processed 288/3084
Processed 320/3084
Processed 352/3084
Processed 384/3084
Processed 416/3084
Processed 448/3084
Processed 480/3084
Processed 512/3084
Processed 544/3084
Processed 576/3084
Processed 608/3084
Processed 640/3084
Processed 672/3084
Processed 704/3084
Processed 736/3084
Processed 768/3084
Processed 800/3084
Processed 832/3084
Processed 864/3084
Processed 896/3084
Processed 928/3084
Processed 960/3084
Processed 992/3084
Processed 1024/3084
Processed 1056/3084
Processed 1088/3084
Processed 1120/3084
Processed 1152/3084
Processed 1184/3084
Processed 1216/3084
Processed 1248/3084
Processed 1280/3084
Processed 1312/3084
Processed 1344/3084
Processed 1376/3084
Processed 1408/3084
Processed 1440/3084
Processed 1472/3084
Processed 1504/3084
Processed 1536/3084
Processed 1568/3084
Processed 1600/3084
Processed 1632/3084
Processed 1664

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# Scale the numeric features so they're on a similar range to each other
numeric_features = model_data[["share_count", "ad_network_score", "clickbait_score"]].values
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

# Fuse: BERT embeddings (768) + economic/behavioural features (3) = 771 total
X = np.hstack([bert_embeddings, numeric_scaled])
y = model_data["label"].values

print("Final feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

# Split into train/test, stratified so both sets keep the same fake/real ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Final feature matrix shape: (3084, 771)
Labels shape: (3084,)
Train: (2467, 771) Test: (617, 771)


In [7]:
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Balance the training set only (never apply SMOTE to test data — that would leak information)
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print("Before SMOTE:", X_train.shape, "| After SMOTE:", X_train_balanced.shape)
print("Balanced label counts:", np.bincount(y_train_balanced))

# Reshape for Conv1D: (samples, features, 1)
X_train_cnn = X_train_balanced.reshape(X_train_balanced.shape[0], X_train_balanced.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Build the CNN
model = Sequential([
    Conv1D(64, kernel_size=3, activation="relu", input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(32, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    X_train_cnn, y_train_balanced,
    validation_split=0.15,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Before SMOTE: (2467, 771) | After SMOTE: (3858, 771)
Balanced label counts: [1929 1929]


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 769, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 384, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 382, 32)        │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 191, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6112)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       391,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 397,729 (1.52 MB)

 Trainable params: 397,729 (1.52 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.6691 - loss: 0.6112 - val_accuracy: 0.4784 - val_loss: 0.8235
Epoch 2/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7682 - loss: 0.5004 - val_accuracy: 0.8359 - val_loss: 0.3933
Epoch 3/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8042 - loss: 0.4340 - val_accuracy: 0.5544 - val_loss: 0.7291
Epoch 4/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8402 - loss: 0.3773 - val_accuracy: 0.8273 - val_loss: 0.4267
Epoch 5/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8628 - loss: 0.3235 - val_accuracy: 0.9067 - val_loss: 0.2597
Epoch 6/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8905 - loss: 0.2782 - val_accuracy: 0.9568 - val_loss: 0.1685
Epoch 7/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8945 - loss: 0.2655 - val_accuracy: 0.9171 - val_loss: 0.2315
Epoch 8/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9109 - loss: 0.2203 - val_accuracy: 0

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Get predictions on the test set (which the model has never seen)
y_pred_proba = model.predict(X_test_cnn)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

print("=== BERT-CNN Hybrid Model — Test Set Performance ===")
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))
print("AUC-ROC:  ", roc_auc_score(y_test, y_pred_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step
=== BERT-CNN Hybrid Model — Test Set Performance ===
Accuracy:  0.8038897893030794
Precision: 0.54421768707483
Recall:    0.5970149253731343
F1-score:  0.5693950177935944
AUC-ROC:   0.8022928834090418

Confusion Matrix:
[[416  67]
 [ 54  80]]

Full classification report:
              precision    recall  f1-score   support

        Real       0.89      0.86      0.87       483
        Fake       0.54      0.60      0.57       134

    accuracy                           0.80       617
   macro avg       0.71      0.73      0.72       617
weighted avg       0.81      0.80      0.81       617



In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split as tts

# Use the SAME train/test split (by index) so comparisons are fair
train_idx, test_idx = train_test_split(
    model_data.index, test_size=0.2, random_state=42, stratify=model_data["label"]
)

titles_train = model_data.loc[train_idx, "title"]
titles_test = model_data.loc[test_idx, "title"]
y_train_txt = model_data.loc[train_idx, "label"]
y_test_txt = model_data.loc[test_idx, "label"]

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(titles_train)
X_test_tfidf = tfidf.transform(titles_test)

results = []

for name, clf in [("Naive Bayes (text-only)", MultinomialNB()),
                   ("SVM (text-only)", LinearSVC())]:
    clf.fit(X_train_tfidf, y_train_txt)
    preds = clf.predict(X_test_tfidf)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_txt, preds),
        "Precision": precision_score(y_test_txt, preds),
        "Recall": recall_score(y_test_txt, preds),
        "F1": f1_score(y_test_txt, preds),
    })

import pandas as pd
print(pd.DataFrame(results))

                     Model  Accuracy  Precision    Recall        F1
0  Naive Bayes (text-only)  0.807131   0.857143  0.134328  0.232258
1          SVM (text-only)  0.833063   0.647619  0.507463  0.569038


In [11]:
def build_and_train_cnn(X_train_feat, y_train_feat, X_test_feat, input_label):
    sm = SMOTE(random_state=42)
    X_bal, y_bal = sm.fit_resample(X_train_feat, y_train_feat)

    n_features = X_bal.shape[1]

    if n_features <= 10:
        # Too few features for CNN's conv+pooling layers — use a small dense network instead
        m = Sequential([
            Dense(16, activation="relu", input_shape=(n_features,)),
            Dropout(0.3),
            Dense(8, activation="relu"),
            Dense(1, activation="sigmoid"),
        ])
        X_bal_in = X_bal
        X_test_in = X_test_feat
    else:
        X_bal_in = X_bal.reshape(X_bal.shape[0], X_bal.shape[1], 1)
        X_test_in = X_test_feat.reshape(X_test_feat.shape[0], X_test_feat.shape[1], 1)
        m = Sequential([
            Conv1D(64, kernel_size=3, activation="relu", input_shape=(X_bal_in.shape[1], 1)),
            MaxPooling1D(pool_size=2),
            Conv1D(32, kernel_size=3, activation="relu"),
            MaxPooling1D(pool_size=2),
            Flatten(),
            Dense(64, activation="relu"),
            Dropout(0.3),
            Dense(1, activation="sigmoid"),
        ])

    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(X_bal_in, y_bal, validation_split=0.15, epochs=20, batch_size=32,
          callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)], verbose=0)

    preds_proba = m.predict(X_test_in, verbose=0)
    preds = (preds_proba > 0.5).astype(int).flatten()

    return {
        "Model": input_label,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "AUC-ROC": roc_auc_score(y_test, preds_proba),
    }

ablation_results = []

ablation_results.append(build_and_train_cnn(X_train[:, :768], y_train, X_test[:, :768], "Linguistic only (BERT)"))
ablation_results.append(build_and_train_cnn(X_train[:, 768:], y_train, X_test[:, 768:], "Economic + Behavioural only"))
ablation_results.append(build_and_train_cnn(X_train, y_train, X_test, "Full fusion (Linguistic + Economic + Behavioural)"))

print(pd.DataFrame(ablation_results))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_r

                                               Model  Accuracy  Precision  \
0                             Linguistic only (BERT)  0.761750   0.466667   
1                        Economic + Behavioural only  0.779579   0.491803   
2  Full fusion (Linguistic + Economic + Behavioural)  0.816856   0.581395   

     Recall        F1   AUC-ROC  
0  0.679104  0.553191  0.790241  
1  0.447761  0.468750  0.676694  
2  0.559701  0.570342  0.794119  
